In [1]:
from pyspark.sql.functions import col, to_timestamp
from pyspark.sql import SparkSession

# spark=SparkSession.builder.appName("duplicate").getOrCreate()
# spark = (
#     SparkSession.builder
#     .appName("LocalSparkTest")
#     .master("local[*]")
#     .config("spark.driver.host", "127.0.0.1")
#     .config("spark.driver.bindAddress", "127.0.0.1")
#     .getOrCreate()
# )
# try:
#     spark.stop()
# except:
#     pass

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("LocalSparkTest")
    .master("local[1]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", "1")
    .getOrCreate()
)

spark.range(5).show()

spark.range(5).show()

data = [
    (1, "Alice", "alice_old@example.com", "2026-01-01 09:00:00"),
    (1, "Alice", "alice_new@example.com", "2026-01-05 10:00:00"),
    (2, "Bob", "bob_old@example.com", "2026-01-02 11:00:00"),
    (2, "Bob", "bob_new@example.com", "2026-01-06 12:00:00"),
    (3, "Charlie", "charlie@example.com", "2026-01-03 13:00:00"),
    (4, "David", "david_old@example.com", "2026-01-01 08:00:00"),
    (4, "David", "david_new@example.com", "2026-01-07 15:00:00")
]

columns = ["user_id", "user_name", "email", "updated_at"]

users_df = spark.createDataFrame(data, columns)

users_df = users_df.withColumn(
    "updated_at",
    to_timestamp(col("updated_at"))
)

users_df.createOrReplaceTempView("users_raw")

users_df.show(truncate=False)

26/06/13 08:14:50 WARN Utils: Your hostname, MKHOMBP14018.local resolves to a loopback address: 127.0.0.1; using 10.10.10.172 instead (on interface en0)
26/06/13 08:14:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/13 08:14:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+

+-------+---------+---------------------+-------------------+
|user_id|user_name|email                |updated_at         |
+-------+---------+---------------------+-------------------+
|1      |Alice    |alice_old@example.com|2026-01-01 09:00:00|
|1      |Alice    |alice_new@example.com|2026-01-05 10:00:00|
|2      |Bob      |bob_old@example.com  |2026-01-02 11:00:00|
|2      |Bob      |bob_new@example.com  |2026-01-06 12:00:00|
|3      |Charlie  |charlie@example.com  |2026-01-03 13:00:00|
|4      |David    |david_old@example.com|2026-01-01 08:00:00|
|4      |David    |david_new@example.com|2026-01-07 15:00:00|
+-------+---------+---------------------+-------------------+



In [2]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = (
    Window
    .partitionBy("user_id")
    .orderBy(col("updated_at").desc())
)

deduped_df = (
    users_df
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)

deduped_df.show(truncate=False)

+-------+---------+---------------------+-------------------+
|user_id|user_name|email                |updated_at         |
+-------+---------+---------------------+-------------------+
|1      |Alice    |alice_new@example.com|2026-01-05 10:00:00|
|2      |Bob      |bob_new@example.com  |2026-01-06 12:00:00|
|3      |Charlie  |charlie@example.com  |2026-01-03 13:00:00|
|4      |David    |david_new@example.com|2026-01-07 15:00:00|
+-------+---------+---------------------+-------------------+



In [3]:
users_df.createOrReplaceTempView("users_raw")

deduped_sql_df = spark.sql("""
WITH ranked AS (
    SELECT
        user_id,
        user_name,
        email,
        updated_at,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY updated_at DESC
        ) AS rn
    FROM users_raw
)

SELECT
    user_id,
    user_name,
    email,
    updated_at
FROM ranked
WHERE rn = 1
""")

deduped_sql_df.show(truncate=False)

+-------+---------+---------------------+-------------------+
|user_id|user_name|email                |updated_at         |
+-------+---------+---------------------+-------------------+
|1      |Alice    |alice_new@example.com|2026-01-05 10:00:00|
|2      |Bob      |bob_new@example.com  |2026-01-06 12:00:00|
|3      |Charlie  |charlie@example.com  |2026-01-03 13:00:00|
|4      |David    |david_new@example.com|2026-01-07 15:00:00|
+-------+---------+---------------------+-------------------+



In [4]:
from pyspark.sql.functions import col, to_timestamp

data = [
    (1, "Alice", "alice_old@example.com", "2026-01-01 09:00:00"),
    (1, "Alice", "alice_new@example.com", "2026-01-05 10:00:00"),
    (2, "Bob", "bob_old@example.com", "2026-01-02 11:00:00"),
    (2, "Bob", "bob_new@example.com", "2026-01-06 12:00:00"),
    (3, "Charlie", "charlie@example.com", "2026-01-03 13:00:00"),
    (4, "David", "david_old@example.com", "2026-01-01 08:00:00"),
    (4, "David", "david_new@example.com", "2026-01-07 15:00:00")
]

columns = ["user_id", "user_name", "email", "updated_at"]

users_df = spark.createDataFrame(data, columns)

users_df = users_df.withColumn(
    "updated_at",
    to_timestamp(col("updated_at"))
)

users_df.createOrReplaceTempView("users_raw")

print("Original data")
users_df.show(truncate=False)

deduped_sql_df = spark.sql("""
WITH ranked AS (
    SELECT
        user_id,
        user_name,
        email,
        updated_at,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY updated_at DESC
        ) AS rn
    FROM users_raw
)

SELECT
    user_id,
    user_name,
    email,
    updated_at
FROM ranked
WHERE rn = 1
""")

print("Latest record per user using SQL CTE")
deduped_sql_df.show(truncate=False)


Original data
+-------+---------+---------------------+-------------------+
|user_id|user_name|email                |updated_at         |
+-------+---------+---------------------+-------------------+
|1      |Alice    |alice_old@example.com|2026-01-01 09:00:00|
|1      |Alice    |alice_new@example.com|2026-01-05 10:00:00|
|2      |Bob      |bob_old@example.com  |2026-01-02 11:00:00|
|2      |Bob      |bob_new@example.com  |2026-01-06 12:00:00|
|3      |Charlie  |charlie@example.com  |2026-01-03 13:00:00|
|4      |David    |david_old@example.com|2026-01-01 08:00:00|
|4      |David    |david_new@example.com|2026-01-07 15:00:00|
+-------+---------+---------------------+-------------------+

Latest record per user using SQL CTE
+-------+---------+---------------------+-------------------+
|user_id|user_name|email                |updated_at         |
+-------+---------+---------------------+-------------------+
|1      |Alice    |alice_new@example.com|2026-01-05 10:00:00|
|2      |Bob      

In [5]:
#Find records where the same email maps to multiple user_ids.

In [6]:
from pyspark.sql.functions import col, lower, trim

data = [
    (1, "Alice", "alice@test.com"),
    (2, "Bob", "bob@test.com"),
    (3, "Charlie", "alice@test.com"),   # same email as Alice, different user_id
    (4, "David", "david@test.com"),
    (5, "Eva", "bob@test.com"),        # same email as Bob, different user_id
    (6, "Frank", "frank@test.com"),
    (7, "Grace", "ALICE@test.com")     # same email but different case
]

columns = ["user_id", "user_name", "email"]

users_df = spark.createDataFrame(data, columns)

users_df = users_df.withColumn(
    "email_clean",
    lower(trim(col("email")))
)


users_df.show(truncate=False)

+-------+---------+--------------+--------------+
|user_id|user_name|email         |email_clean   |
+-------+---------+--------------+--------------+
|1      |Alice    |alice@test.com|alice@test.com|
|2      |Bob      |bob@test.com  |bob@test.com  |
|3      |Charlie  |alice@test.com|alice@test.com|
|4      |David    |david@test.com|david@test.com|
|5      |Eva      |bob@test.com  |bob@test.com  |
|6      |Frank    |frank@test.com|frank@test.com|
|7      |Grace    |ALICE@test.com|alice@test.com|
+-------+---------+--------------+--------------+



In [7]:
users_df.createOrReplaceTempView("users")


In [13]:
spark.sql(

"""
with temp_use as (
select email,COUNT(user_id) as cnt  from users group by email
)
select * from temp_use

"""
    
).show()

+--------------+---+
|         email|cnt|
+--------------+---+
|alice@test.com|  2|
|  bob@test.com|  2|
|david@test.com|  1|
|frank@test.com|  1|
|ALICE@test.com|  1|
+--------------+---+

